---
# PARTIE 2 -- Structured Streaming (après-midi)

## 2.1 Concepts fondamentaux

### Du batch au streaming

Le batch traite un ensemble de données **figé** : on sait à l'avance combien
il y a de lignes, on peut trier, on peut faire des jointures complètes.

Le streaming traite un flux de données **continu** : les données arrivent
en permanence, on ne connaît pas la fin, et certains événements peuvent
arriver en retard.

Spark Structured Streaming répond à ce problème avec un modèle élégant :
le flux est traité comme une **table infinie** qui s'agrandit au fil du temps.
Les requêtes SQL ou DataFrame s'y appliquent sans modification.

```
Flux d'entrée  :  [evt1] [evt2] [evt3] ...  [evtN]  ...
                     |      |      |              |
                  +--+------+------+--------------+----→ temps
                  |
                  Spark : exécution de micro-batchs toutes les X secondes
                  |
                  Résultat : table de résultats mise à jour en continu
```

### Vocabulaire

| Terme | Définition |
|-------|-----------|
| **Trigger** | Fréquence d'exécution des micro-batchs |
| **Checkpoint** | Sauvegarde de l'état pour la reprise après panne |
| **Watermark** | Délai maximal toléré pour les données tardives |
| **Fenêtre glissante** | Agrégation sur une plage de temps mobile |
| **Sink** | Destination de l'écriture (console, fichier, Delta...) |


---
## 2.2 Le simulateur de flux

En production, le flux proviendrait de Kafka ou de l'API GBFS en direct.
Pour ce cours, un script Python **rejoue les données historiques** en écrivant
des fichiers JSON dans un répertoire surveillé par Spark.

Ce mécanisme -- la **file source** (file source) -- est le moyen le plus simple
de tester Structured Streaming sans infrastructure externe.

> Le simulateur est fourni dans `scripts/simulateur_flux.py`.
> Lancez-le dans un terminal séparé **avant** d'exécuter les cellules suivantes.
>
> ```bash
> python scripts/simulateur_flux.py --output data/output/stream_input --vitesse 3
> ```
>
> L'option `--vitesse 3` signifie que chaque seconde réelle correspond à
> 3 minutes de données historiques.


### Lancement automatique du simulateur (ajout du groupe)

L'énoncé original demande de lancer `scripts/simulateur_flux.py` "à la main"
dans un terminal séparé avant d'exécuter les cellules qui suivent. Ce script
ne figurait pas parmi les fichiers fournis : nous l'avons écrit nous-mêmes
(voir `scripts/simulateur_flux.py`, docstring en tête de fichier pour le
détail) en respectant le contrat attendu par le schéma `schema_flux` plus
bas.

Pour que le notebook reste **reproductible en une seule exécution "Run All"**
(contrainte n°2 du projet), nous le lançons ici directement comme
sous-processus Python en arrière-plan, plutôt que d'exiger une étape
manuelle dans un second terminal. Il est arrêté proprement à la toute fin du
notebook (section 2.8).

In [ ]:
import subprocess
import sys

# Lance le simulateur en arrière-plan (processus détaché du noyau Jupyter).
# --intervalle-sec 2 : un nouveau micro-batch toutes les 2 secondes réelles,
#   ajusté pour rester cohérent avec les triggers resserrés de ce notebook
#   (cf. note de la section 2.5) sans avoir à attendre plusieurs minutes.
# --max-fichiers 0   : tourne sans limite, on l'arrêtera explicitement en fin
#   de notebook (section 2.8) via proc_simulateur.terminate().
proc_simulateur = subprocess.Popen(
    [
        sys.executable, "../scripts/simulateur_flux.py",
        "--output", str(STREAM_SOURCE_DIR),
        "--n-stations", "60",
        "--vitesse", "3",
        "--intervalle-sec", "2",
        "--max-fichiers", "0",
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)
print(f"Simulateur de flux démarré en arrière-plan (PID {proc_simulateur.pid}).")
print(f"Répertoire surveillé : {STREAM_SOURCE_DIR}")

In [ ]:
# Vérification : le simulateur tourne-t-il ?
import time, os

def compter_fichiers_stream(attente_sec: int = 5) -> int:
    """Attend et compte les fichiers JSON produits par le simulateur.

    Args:
        attente_sec: Nombre de secondes à attendre avant de recompter (laisse
            le temps au simulateur d'écrire au moins un fichier).

    Returns:
        Nombre de fichiers .json actuellement présents dans STREAM_SOURCE_DIR.
    """
    time.sleep(attente_sec)
    fichiers = list(STREAM_SOURCE_DIR.glob("*.json"))
    return len(fichiers)

n = compter_fichiers_stream(3)
if n == 0:
    print("[ATTENTION] Aucun fichier JSON trouvé dans le répertoire de streaming.")
    print(f"  Vérifiez que le simulateur tourne dans un terminal séparé.")
    print(f"  Répertoire surveillé : {STREAM_SOURCE_DIR}")
else:
    dernier = sorted(STREAM_SOURCE_DIR.glob("*.json"))[-1]
    print(f"{n} fichier(s) JSON déjà présents. Dernier fichier : {dernier.name}")
    print(dernier.read_text(encoding="utf-8")[:400])

---
## 2.3 Source de streaming : `readStream`

`spark.readStream` fonctionne exactement comme `spark.read`, avec deux différences :

1. Il retourne un **DataFrame de streaming** (pas un DataFrame ordinaire).
2. Il ne peut pas être affiché directement avec `.show()` -- il faut déclencher
   une **requête de streaming** avec `.writeStream`.


In [ ]:
# Schéma du flux JSON produit par le simulateur
schema_flux = StructType([
    StructField("station_id",       IntegerType(),   False),
    StructField("nom_station",      StringType(),    True),
    StructField("code_arr",         IntegerType(),   True),
    StructField("capacite",         IntegerType(),   True),
    StructField("velos_meca",       IntegerType(),   True),
    StructField("velos_elec",       IntegerType(),   True),
    StructField("bornettes_libres", IntegerType(),   True),
    StructField("horodatage",       TimestampType(), False),   # timestamp parsé
])

# Création du DataFrame de streaming
# au plus 2 fichiers par micro-batch
# Un schéma explicite est OBLIGATOIRE en Structured Streaming (contrairement
# au batch) : Spark ne peut pas se permettre d'inférer le schéma en relisant
# tout le flux à chaque micro-batch.
stream_df = (
    spark.readStream
    .schema(schema_flux)
    .option("maxFilesPerTrigger", 2)
    .json(str(STREAM_SOURCE_DIR))
)

print(f"Est un streaming DataFrame : {stream_df.isStreaming}")
print(f"Colonnes : {stream_df.columns}")
# On ne peut pas appeler .count() ou .show() sur un streaming DataFrame

In [ ]:
# Ajout des colonnes calculées -- exactement comme en batch
# C'est le point clé de l'API Structured Streaming : les mêmes opérations
# DataFrame (withColumn, filter, groupBy...) s'appliquent indifféremment à
# un DataFrame batch ou streaming -- Spark se charge de la ré-exécution
# incrémentale à chaque micro-batch.
stream_enrichi = (
    stream_df
    .withColumn("velos_total", F.col("velos_meca") + F.col("velos_elec"))
    .withColumn(
        "taux_occupation",
        F.round((F.col("velos_meca") + F.col("velos_elec")) / F.col("capacite"), 4)
    )
    .withColumn("est_vide", (F.col("velos_meca") + F.col("velos_elec")) == 0)
)
print("Colonnes après enrichissement :", stream_enrichi.columns)

---
## 2.4 Première requête : sink console

La sortie `console` écrit les résultats dans le terminal Jupyter.
C'est utile pour le débogage -- jamais pour la production.


In [ ]:
# Requête de streaming vers la console
# outputMode :
#   "append"  -- écrit uniquement les nouvelles lignes (défaut pour les flux sans agrégation)
#   "complete" -- réécrit l'intégralité du résultat à chaque batch
#   "update"  -- écrit uniquement les lignes qui ont changé
#
# La fonction start() lance le streaming

q_console = (
    stream_enrichi
    .select("station_id", "nom_station", "code_arr", "horodatage",
            "velos_total", "taux_occupation", "est_vide")
    .writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .option("numRows", 10)
    .trigger(processingTime="3 seconds")
    .start()
)

# La requête tourne en arrière-plan.
# awaitTermination() bloquerait indéfiniment -- on attend juste quelques batchs.
# NB : la durée d'attente ci-dessous (12 s) a été réduite par rapport à la
# version originale de l'énoncé (20 s) pour que ce notebook se rejoue
# rapidement lors de la correction, tout en laissant le temps à 2-3 micro-batchs
# de s'exécuter avec un trigger de 3 s -- voir la note de bas de section.
time.sleep(12)
print(f"Statut de la requête : {q_console.status}")
print(f"Batchs traités : {q_console.lastProgress['numInputRows'] if q_console.lastProgress else 'N/A'}")
q_console.stop()
print("Requête console arrêtée.")

---
## 2.5 Agrégations sur fenêtres temporelles glissantes

L'agrégation sur des **fenêtres temporelles** est la requête streaming la plus
courante en pratique. Elle répond à des questions du type :
"Combien de vélos sont disponibles par arrondissement sur les 10 dernières minutes ?"

Spark propose deux types de fenêtres temporelles :

| Type | Définition | Exemple |
|------|-----------|---------|
| Fenêtre **basculante** | Fenêtres fixes, sans chevauchement | 0-10 min, 10-20 min... |
| Fenêtre **glissante** | Fenêtres qui se chevauchent | [0-10], [5-15], [10-20]... |

```
Fenêtre basculante (10 min)  : |--w1--|--w2--|--w3--|
Fenêtre glissante (10/5)   : |--w1--|          Taille=10, pas=5
                                  |--w2--|
                                       |--w3--|
```

> **Note sur les durées de ce notebook (ajout du groupe) :** l'énoncé original
> prévoit des temps d'attente longs (20 à 60 s) pour observer plusieurs
> micro-batchs sur un flux ralenti "en vitesse réelle". Nous avons réduit ces
> attentes et resserré les triggers (3 à 10 s au lieu de 5 à 10 s avec des
> pauses de 20 à 60 s) afin que l'ensemble du notebook -- y compris la partie
> streaming -- reste rejouable en quelques minutes lors de la correction,
> sans changer la logique pédagogique : le nombre de micro-batchs observés
> reste suffisant pour illustrer fenêtres glissantes, sink Delta, alertes et
> watermark.

In [ ]:
from pyspark.sql.functions import window

# Agrégation glissante : disponibilité moyenne par arrondissement
# Fenêtre de 10 minutes, avançant toutes les 2 minutes
# withWatermark() est nécessaire pour permettre à Spark de savoir quand une
# fenêtre peut être considérée comme "fermée" (et donc émise en mode append) :
# sans watermark, une agrégation fenêtrée ne peut pas être écrite en append.
df_fenetre_arr = (
    stream_enrichi
    .withWatermark("horodatage", "5 minutes")
    .groupBy(
        F.window("horodatage", "10 minutes", "2 minutes"),
        "code_arr",
    )
    .agg(
        F.round(F.avg("velos_total"), 2).alias("velos_disponibles_moyen"),
        F.sum("velos_total").alias("velos_disponibles_total"),
        F.count("*").alias("nb_observations"),
    )
    .select(
        F.col("window.start").alias("fenetre_debut"),
        F.col("window.end").alias("fenetre_fin"),
        "code_arr", "velos_disponibles_moyen", "velos_disponibles_total", "nb_observations",
    )
)

print("Schéma de la requête fenêtrée :")
df_fenetre_arr.printSchema()

In [ ]:
# Écriture vers Delta Lake (sink delta) -- mode append
q_fenetres = (
    df_fenetre_arr
    .writeStream
    .format("delta")
    .outputMode("append")
    # Le checkpoint mémorise les offsets déjà traités : indispensable pour
    # reprendre une requête de streaming après un redémarrage sans dupliquer
    # ni perdre de données ("exactly-once" côté source fichier + sink Delta).
    .option("checkpointLocation", str(STREAM_CHECKPOINT / "fenetres_arrondissement"))
    .trigger(processingTime="5 seconds")
    .start(str(DELTA_DISPONIBLE.parent / "fenetres_arrondissement"))
)

print(f"Requête démarrée : {q_fenetres.name}")
print(f"ID               : {q_fenetres.id}")
print("En attente de plusieurs micro-batchs...")
time.sleep(20)
print(f"Dernier progrès : {q_fenetres.lastProgress}")

In [ ]:
# Lecture des résultats accumulés dans Delta
path_fenetres = str(DELTA_DISPONIBLE.parent / "fenetres_arrondissement")

try:
    # Lire les données au format delta, trier par début de fenêtre et en afficher 30
    spark.read.format("delta").load(path_fenetres).orderBy("fenetre_debut").show(30, truncate=False)
except Exception as e:
    print(f"Pas encore de données : {e}")
    print("Attendez encore quelques secondes et relancez cette cellule.")

---
## 2.6 Alertes : détection de stations en rupture prolongée

L'objectif opérationnel est d'alerter l'équipe de maintenance quand une station
reste vide (zéro vélo disponible) pendant au moins **deux fenêtres consécutives**,
soit 20 minutes de rupture continue.

### Approche : `foreachBatch`

Pour une logique d'alerte complexe (état entre batchs, seuil consécutif),
`foreachBatch` est plus adapté que les agrégations pures. Il permet d'exécuter
une fonction Python arbitraire sur chaque micro-batch.


In [ ]:
# Table d'alertes : schema
schema_alertes = StructType([
    StructField("station_id",   IntegerType(),   False),
    StructField("nom_station",  StringType(),    True),
    StructField("code_arr",     IntegerType(),   True),
    StructField("debut_rupture",TimestampType(), True),
    StructField("fin_rupture",  TimestampType(), True),
    StructField("duree_min",    IntegerType(),   True),
    StructField("ts_alerte",    TimestampType(), True),
])

# Initialisation de la table Delta d'alertes (vide)
# On crée un DataFrame vide MAIS typé avec le schéma cible : cela fixe le
# schéma Delta dès la première écriture, ce que `foreachBatch` viendra
# ensuite alimenter en mode append micro-batch par micro-batch.
df_alertes_vide = spark.createDataFrame([], schema_alertes)
(
    df_alertes_vide
    .write
    .format("delta")
    .mode("overwrite")
    .save(str(DELTA_ALERTES))
)
print(f"Table d'alertes initialisée : {DELTA_ALERTES}")

In [ ]:
from delta.tables import DeltaTable
from datetime import datetime, timezone

# Mémoire inter-batchs : stations actuellement en rupture et depuis quand
etat_ruptures: dict[int, dict] = {}

def traiter_batch_alertes(batch_df, batch_id: int) -> None:
    """Fonction appelée par foreachBatch pour chaque micro-batch.

    Détecte les stations qui passent en rupture ou sortent de rupture.
    Écrit les alertes complètes dans la table Delta.

    Args:
        batch_df : DataFrame du micro-batch courant.
        batch_id : Identifiant séquentiel du batch.
    """
    global etat_ruptures

    # Sortir si `batch_df` est vide
    # isEmpty() est une action peu coûteuse (s'arrête dès la première ligne
    # trouvée) -- on évite ainsi de lancer un job d'agrégation pour rien.
    if batch_df.rdd.isEmpty():
        return

    # Agrégation au niveau station sur ce batch
    # On souhaite collecter :
    # - l'horodatage le plus ancien
    # - l'horodatage le plus récent
    # - le nombre d'observations où la station est vide
    # - le nombre total de stations
    # - le code de l'arrondissement
    # - le nom de la station
    etat_batch = (
        batch_df
        .groupBy("station_id", "nom_station", "code_arr")
        .agg(
            F.min("horodatage").alias("ts_min"),
            F.max("horodatage").alias("ts_max"),
            F.sum(F.when(F.col("est_vide"), 1).otherwise(0)).alias("nb_vide"),
            F.count("*").alias("nb_total"),
        )
        .collect()   # micro-batch de taille bornée -> collect() raisonnable ici
    )

    alertes = []
    # Examiner les transitions :
    # - non vide -> vide
    # - vide -> non vide
    # Dans le premier cas, ajouter la les informations dela station dans `etat_rupture`.
    # Dans le second cas :
    # - calculer le temps pendant le quel la station a été en rupture
    # - si ce temps est > 10 minutes, ajouter une alerte avec les informations détailllées
    # - retirer la station de la liste `etat_rupture`
    for row in etat_batch:
        station_id = row["station_id"]
        # On considère la station "en rupture sur ce batch" si TOUTES ses
        # observations du micro-batch la montrent vide (signal robuste au bruit).
        station_vide_ce_batch = row["nb_total"] > 0 and row["nb_vide"] == row["nb_total"]

        if station_vide_ce_batch and station_id not in etat_ruptures:
            # Transition non-vide -> vide : on commence à suivre la rupture
            etat_ruptures[station_id] = {
                "debut": row["ts_min"],
                "nom_station": row["nom_station"],
                "code_arr": row["code_arr"],
            }
        elif not station_vide_ce_batch and station_id in etat_ruptures:
            # Transition vide -> non-vide : la rupture est terminée
            debut = etat_ruptures[station_id]["debut"]
            fin = row["ts_min"]
            duree_min = int((fin - debut).total_seconds() / 60)
            if duree_min > 10:
                alertes.append({
                    "station_id": station_id,
                    "nom_station": etat_ruptures[station_id]["nom_station"],
                    "code_arr": etat_ruptures[station_id]["code_arr"],
                    "debut_rupture": debut,
                    "fin_rupture": fin,
                    "duree_min": duree_min,
                    "ts_alerte": datetime.now(timezone.utc),
                })
            del etat_ruptures[station_id]

    # Si des alertes ont été enregistrées,
    # les sauvegarder au format `delta` dans DELTA_ALERTES
    # Laisser une trace dans les logs
    if alertes:
        df_alertes_batch = spark.createDataFrame(alertes, schema=schema_alertes)
        (
            df_alertes_batch
            .write
            .format("delta")
            .mode("append")
            .save(str(DELTA_ALERTES))
        )
        print(f"[batch {batch_id}] {len(alertes)} nouvelle(s) alerte(s) écrite(s)")
    else:
        print(f"[batch {batch_id}] aucune transition de rupture détectée "
              f"({len(etat_ruptures)} station(s) actuellement en rupture)")

In [ ]:
# Lancement de la requête d'alerte
# - avec une tolérance de 5 minutes sur l'horodatage (cf. section suivante)
# - qui exécute la fonction `traiter_batch_alertes` pour chaque batch
# - définissant un `checkpoint` pour la sauvegarde de l'état en cas de panne
# - la requête s'appelle `q_alertes_rupture`
# - avec un intervalle de traitement de 10 ssecondes
q_alertes = (
    stream_enrichi
    .withWatermark("horodatage", "5 minutes")
    .writeStream
    .foreachBatch(traiter_batch_alertes)
    .option("checkpointLocation", str(STREAM_CHECKPOINT / "alertes_rupture"))
    .queryName("q_alertes_rupture")
    .trigger(processingTime="10 seconds")
    .start()
)

print("Requête d'alertes démarrée. En attente de données...")
time.sleep(25)   # réduit par rapport à l'énoncé original (60 s) -- cf. note plus haut

# Lecture des alertes produites
df_alertes = spark.read.format("delta").load(str(DELTA_ALERTES))
print(f"\nAlertes enregistrées : {df_alertes.count()}")
df_alertes.orderBy(F.desc("ts_alerte")).show(20, truncate=False)

---
## 2.7 Données tardives et watermark

Dans un système distribué, certains événements arrivent avec du retard
(réseau saturé, capteur hors ligne temporairement...). Spark doit décider
combien de temps attendre avant de considérer une fenêtre comme fermée.

C'est le rôle du **watermark** : il définit le retard maximal toléré.

```
Watermark de 5 minutes :
  Si le timestamp max observé est 10h30, Spark considère que
  toutes les données avant 10h25 sont arrivées.
  Les fenêtres fermées avant 10h25 ne seront plus mises à jour.
```

### Illustration avec des données tardives simulées


In [ ]:
import json, datetime as dt

# Injection de données tardives :
# Écrire dans le répertoire de streaming un fichier avec des horodatages "dans le passé"
# Les dates sont dans le format ISO
# Le fichier est injecté dans le répertroie surveillé par Spark.
# Chaque line est homogène à une observation (ou snapshot).
# N.B. Ajouter ne migne dans le fichier de logs
def injecter_donnees_tardives(retard_minutes: int, nb_lignes: int = 20):
    """Écrit un fichier JSON avec des horodatages retardés.

    Args:
        retard_minutes : Retard à simuler (en minutes).
        nb_lignes      : Nombre de snapshots à générer.
    """
    instant = dt.datetime.now(dt.timezone.utc) - dt.timedelta(minutes=retard_minutes)
    lignes = []
    for i in range(nb_lignes):
        capacite = 25
        velos_total = (i * 3) % (capacite + 1)   # valeurs déterministes mais variées
        lignes.append({
            "station_id": 1000 + (i % 60),
            "nom_station": f"Station {i % 60:03d} (tardif)",
            "code_arr": 75001 + (i % 20),
            "capacite": capacite,
            "velos_meca": velos_total // 2,
            "velos_elec": velos_total - velos_total // 2,
            "bornettes_libres": capacite - velos_total,
            "horodatage": instant.strftime("%Y-%m-%dT%H:%M:%S+00:00"),
        })

    chemin = STREAM_SOURCE_DIR / f"tardif_retard{retard_minutes}min_{instant.strftime('%H%M%S')}.json"
    chemin_tmp = chemin.with_suffix(".json.tmp")
    with open(chemin_tmp, "w", encoding="utf-8") as f:
        json.dump(lignes, f)
    chemin_tmp.rename(chemin)   # écriture atomique, comme dans le simulateur
    print(f"  Fichier tardif écrit : {chemin.name} (retard {retard_minutes} min, {nb_lignes} lignes)")

# Scénario 1 : retard de 3 min -- dans le watermark (5 min) -> données prises en compte
injecter_donnees_tardives(retard_minutes=3, nb_lignes=10)
time.sleep(6)

# Scénario 2 : retard de 12 min -- hors watermark -> données ignorées par les agrégations
injecter_donnees_tardives(retard_minutes=12, nb_lignes=10)
time.sleep(6)

print("\nStatut des requêtes actives :")
for q in spark.streams.active:
    prog = q.lastProgress
    print(f"  {q.name:<30} -- inputRows : {prog['numInputRows'] if prog else 'N/A'}")

In [ ]:
# [EXERCICE]
# Créez une nouvelle requête de streaming qui calcule,
# pour chaque arrondissement et sur une fenêtre basculante de 15 minutes,
# le MAXIMUM du taux_occupation observé.
# Écrivez le résultat en mode "update" vers la console.
#
# Rappel : fenêtre basculante = window(col, "15 minutes")  (sans 3e argument)
# ──────────────────────────────────────────────────────────────────────────

# Votre code ici :
q_exercice = (
    stream_enrichi
    .withWatermark("horodatage", "5 minutes")
    .groupBy(F.window("horodatage", "15 minutes"), "code_arr")
    .agg(F.max("taux_occupation").alias("taux_max"))
    .writeStream
    .format("console")
    .outputMode("update")
    .option("truncate", False)
    .trigger(processingTime="5 seconds")
    .start()
)
time.sleep(8)
q_exercice.stop()
print("Requête d'exercice arrêtée.")

---
## 2.8 Arrêt propre des requêtes et bilan

En production, les requêtes de streaming tournent indéfiniment.
Dans un contexte de cours, il faut les arrêter proprement pour libérer les ressources.


In [ ]:
# Arrêt de toutes les requêtes actives
actives = spark.streams.active
print(f"Requêtes actives à arrêter : {[q.name for q in actives]}")

# Arrêter proprement les requêtes et reseigner le fichier logs
for q in actives:
    nom = q.name or q.id
    q.stop()
    print(f"  Requête '{nom}' arrêtée proprement.")

print("\nToutes les requêtes de streaming sont arrêtées.")

# Arrêt du simulateur de flux lancé en arrière-plan en début de notebook
# (ajout du groupe -- cf. section 2.2)
proc_simulateur.terminate()
proc_simulateur.wait(timeout=5)
print("Simulateur de flux arrêté.")

In [ ]:
# Lecture finale des résultats accumulés
print("=== Résultats finaux ===")

print("\n-- Fenêtres arrondissement --")
try:
    df_fin_fenetres = spark.read.format("delta").load(
        str(DELTA_DISPONIBLE.parent / "fenetres_arrondissement")
    )
    print(f"  {df_fin_fenetres.count()} fenêtres enregistrées")
    df_fin_fenetres.orderBy(F.desc("fenetre_debut")).show(10, truncate=False)
except Exception as e:
    print(f"  Aucun résultat : {e}")

print("\n-- Alertes rupture --")
try:
    df_fin_alertes = spark.read.format("delta").load(str(DELTA_ALERTES))
    print(f"  {df_fin_alertes.count()} alertes enregistrées")
    df_fin_alertes.orderBy(F.desc("ts_alerte")).show(10, truncate=False)
except Exception as e:
    print(f"  Aucune alerte : {e}")


---
## Bilan du Jour 2

### Ce que nous avons fait

| Étape | Module | Concept clé |
|-------|--------|-------------|
| Vues temporaires et SQL de base | Spark SQL | `createOrReplaceTempView`, SQL natif |
| Questions métier analytiques | Spark SQL | `LEFT ANTI JOIN`, `CASE WHEN`, sous-requêtes |
| Fenêtrage analytique | Spark SQL / DataFrame | `OVER`, `LAG`, `LEAD`, `ROW_NUMBER`, `AVG OVER` |
| Écriture Delta Lake | Delta | `format("delta")`, partitionnement |
| Time-travel | Delta | `versionAsOf`, historique des opérations |
| Mise à jour incrémentale | Delta | `MERGE INTO`, `whenMatchedUpdateAll` |
| Source de streaming | Structured Streaming | `readStream`, schéma explicite, file source |
| Sink console | Structured Streaming | débogage, `outputMode("append")` |
| Fenêtres glissantes | Structured Streaming | `window()`, basculante vs glissante |
| Sink Delta | Structured Streaming | écriture transactionnelle en flux |
| Alertes avec `foreachBatch` | Structured Streaming | logique inter-batchs, état partagé |
| Watermark et late data | Structured Streaming | tolérance, fermeture de fenêtres |

### Points d'attention

- En `outputMode("append")`, Spark n'écrit que des lignes **nouvelles et définitives**.
  Pour les agrégations fenêtrées, cela implique un watermark : Spark attend que la fenêtre
  soit fermée avant d'émettre son résultat.
- `foreachBatch` est puissant mais introduit un état mutable (`etat_ruptures`) qui
  n'est **pas persisté dans le checkpoint**. En cas de redémarrage, l'état est perdu.
  Pour un système de production, il faudrait utiliser `mapGroupsWithState` ou
  stocker l'état dans Delta Lake.
- Le checkpoint est **obligatoire** pour toute requête qui ne doit pas repartir de zéro
  après un redémarrage.

### Pour demain (Jour 3)

La table Delta `disponibilite` (batch) et les résultats des fenêtres glissantes
(streaming) serviront de données d'entrée pour le Jour 3 : construction des features,
entraînement d'un modèle de régression avec MLlib, clustering des stations avec K-Means
et suivi des expériences avec MLflow.


In [ ]:
spark.stop()
print("SparkSession arrêtée. À demain !")
